# Figure: rotational invariance is structural

Paper section: *Structural Properties of DIWECT Embeddings*, paragraph *Rotational Invariance is Structural*


In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import common as C

C.use_paper_style()
FOLDER = "1-mechanism"

SIZE, AREA = 64, 380                 # every class has exactly 380 on-pixels
NUM_DIRS, NUM_HEIGHTS = 64, 48       # WECT: 64 x 48 = 3072-dim; DIWECT: 48-dim
ANGLE_STEP = 3.0                     # degrees, for the drift sweep (b)
POSE_STEP = 10.0                     # degrees, for signal vs nuisance (c): 36 steps

transform = C.WECTTransform(num_dirs=NUM_DIRS, num_heights=NUM_HEIGHTS)
shapes = C.make_shape_library(size=SIZE, area=AREA)
print(f"classes: {list(shapes)} (all {int(next(iter(shapes.values())).sum())} on-pixels)")

## (b) Drift under rotation

We wish to measure the relative $L_2$ distance of each descriptor from its own value at 0°, as the L shape turns through a full circle. Our rotated elements snap to the nearest-neighbor. This, we add a control that rotates the polygon and rasterizes afterwards (and then doesn't require resampling). We observe that it follows the same curve as the DIWECT, so the DIWECT's residual drift comes from the rasterized object changing with the angle as opposed to from the descriptor.

In [ ]:
def relative_l2(x, ref):
    return float(np.linalg.norm(np.ravel(x - ref)) / (np.linalg.norm(np.ravel(ref)) + 1e-12))


def rotate_polygon(poly, deg):
    t = np.radians(deg)
    c, s = np.cos(t), np.sin(t)
    return [(x * c - y * s, x * s + y * c) for x, y in poly]


ell = shapes["ell"]
W_ell = transform.wect(ell)       # (D, H)
d_ell = transform.diwect(ell)     # (H,), exact DIWECT (src/diwect.py)
# sanity check: the WECT averaged over the 64 sampled directions approximates it
avg_err = np.abs(W_ell.mean(0) - d_ell).max() / np.abs(d_ell).max()
assert avg_err < 0.05, avg_err
print(f"64-direction averaged WECT vs exact DIWECT: max relative difference {avg_err:.3f}")

angles = np.arange(0.0, 360.0, ANGLE_STEP)
rotated = [C.rotate_image(ell, a) for a in angles]
reraster = [C.make_shape(rotate_polygon(C.SHAPE_POLYGONS["ell"], a), size=SIZE, area=AREA) for a in angles]

drift = pd.DataFrame({
    "angle": angles,
    "wect": [relative_l2(transform.wect(im), W_ell) for im in rotated],
    "diwect": [relative_l2(transform.diwect(im), d_ell) for im in rotated],
    "diwect_reraster": [relative_l2(transform.diwect(im), d_ell) for im in reraster],
})
print(f"mean drift over 360 deg: WECT {drift.wect.mean():.4f}, DIWECT {drift.diwect.mean():.4f} "
      f"({drift.wect.mean() / drift.diwect.mean():.1f}x smaller); "
      f"rotate-then-rasterize control: DIWECT {drift.diwect_reraster.mean():.4f}")

## (c) Signal versus nuisance

Each of the five shapes is rotated in 36 uniform steps of 10°. In each feature space, we normalize by the mean feature norm:

* **pose spread**: mean distance of a rotated sample from its own class mean
* **class separation**: mean distance between class means

In [ ]:
pose_angles = np.arange(0.0, 360.0, POSE_STEP)
feats = {"wect": {}, "diwect": {}}
for name, img in shapes.items():
    rotated_imgs = [C.rotate_image(img, a) for a in pose_angles]
    feats["wect"][name] = np.stack([transform.wect(im).ravel() for im in rotated_imgs])
    feats["diwect"][name] = np.stack([transform.diwect(im) for im in rotated_imgs])

rows = []
for rep in ("wect", "diwect"):
    scale = float(np.mean([np.linalg.norm(feats[rep][k], axis=1).mean() for k in shapes]))
    within = float(np.mean([np.linalg.norm(feats[rep][k] - feats[rep][k].mean(0), axis=1).mean()
                            for k in shapes])) / scale
    between = float(np.mean([np.linalg.norm(feats[rep][a].mean(0) - feats[rep][b].mean(0))
                             for a, b in itertools.combinations(shapes, 2)])) / scale
    rows.append({"representation": rep, "pose_spread": within, "class_separation": between,
                 "ratio": between / within})
sig = pd.DataFrame(rows).set_index("representation")
print(sig.round(3).to_string())

## Figure

In [ ]:
# (a) the five equal-area shapes
fig, axes = plt.subplots(1, len(shapes), figsize=(2.2 * len(shapes), 2.2))
for ax, img in zip(axes, shapes.values()):
    ax.imshow(img, cmap="magma", vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
C.save_pdf(fig, FOLDER, "figure1a-shapes")

# (b) drift under rotation with rotate-then-rasterize control
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.plot(drift.angle, drift.wect, color=C.color("wect"), lw=2, label="WECT")
ax.plot(drift.angle, drift.diwect, color=C.color("diwect"), lw=2, label="DIWECT")
ax.plot(drift.angle, drift.diwect_reraster, color="k", lw=1.1, ls="--",
        label="DIWECT, rotate-then-\nrasterize control")
ax.set_xlim(0, 360)
ax.set_xticks(range(0, 361, 90))
ax.set_xlabel("rotation of the object (degrees)")
ax.set_ylabel("relative $L_2$ drift from 0°")
ax.set_ylim(0, 1.18)
ax.legend(loc="upper center", fontsize=7.5, ncol=1)
C.save_pdf(fig, FOLDER, "figure1b-drift")

# (c) rotation spread (amber) vs class separation (green)
POSE_C, CLASS_C = "#e09f3e", "#4c9f70"
fig, ax = plt.subplots(figsize=(4.2, 3.6))
x = np.arange(2)
b1 = ax.bar(x - 0.19, sig["pose_spread"], 0.36, color=POSE_C, label="rotation spread")
b2 = ax.bar(x + 0.19, sig["class_separation"], 0.36, color=CLASS_C, label="class separation")
C.bar_labels(ax, b1); C.bar_labels(ax, b2)
for i, rep in enumerate(sig.index):
    ax.annotate(f"S/N = {sig.ratio[rep]:.2f}", (i, 0.60), ha="center", fontsize=9.5, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([C.label(r) for r in sig.index])
ax.set_ylim(0, 0.70)
ax.set_ylabel("normalized distance")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
C.save_pdf(fig, FOLDER, "figure1c-signal-vs-nuisance")